In [ ]:
from utils import bootstrap_cloud
bootstrap_cloud()

# Experiment 01 - Baseline CNN

**Goal:** Establish the performance floor with a minimal 2-block CNN, no augmentation.  
**Model:** `BaselineCNN`
**Metric (primary):** Macro-F1 on the held-out test set  
**Augmentation:** None (eval transform only -> resize + normalize)do n  

Results from this notebook define the baseline that all subsequent experiments must beat.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('.').resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import torch
from config import load_config
from augmentation import build_eval_transform
from dataset import build_dataloaders
from model import BaselineCNN
from training import Trainer, compute_class_weights
from utils import set_seed, show_results, load_model

cfg = load_config(ROOT / 'configs/config.yaml')
set_seed(cfg.preprocessing.random_seed, cfg.device)
print(f'Device: {cfg.device} | Classes: {len(cfg.classes)}')

Device: cpu  |  Classes: 37


In [ ]:
EXP_NAME = 'baseline'
NUM_EPOCHS = cfg.training.num_epochs

splits_dir = cfg.splits_dir
dl_cfg = cfg.dataloader

# No augmentation for baseline — pure eval transform on all splits
eval_tf = build_eval_transform(cfg.preprocessing.image_size, use_imagenet_norm=False)

train_loader, val_loader, test_loader = build_dataloaders(
    train_csv=str(splits_dir / 'train.csv'),
    val_csv=str(splits_dir / 'val.csv'),
    test_csv=str(splits_dir / 'test.csv'),
    train_transform=eval_tf,
    eval_transform=eval_tf,
    data_root=str(cfg.data_root),
    batch_size=dl_cfg.batch_size,
    num_workers=dl_cfg.num_workers,
    pin_memory=dl_cfg.pin_memory,
    persistent_workers=dl_cfg.persistent_workers,
    prefetch_factor=dl_cfg.prefetch_factor,
    use_weighted_sampler=False,
)
print(f'Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}  Test: {len(test_loader.dataset):,}')

In [ ]:
num_classes = len(cfg.classes)
model = BaselineCNN(in_channels=3, num_classes=num_classes)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

In [ ]:
class_weights = None
if cfg.training.use_class_weights:
    class_weights = compute_class_weights(splits_dir / 'train.csv', num_classes)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=cfg.device,
    exp_name=EXP_NAME,
    results_root=cfg.results_root,
    checkpoints_root=cfg.checkpoints_root,
    num_epochs=NUM_EPOCHS,
    learning_rate=cfg.training.learning_rate,
    weight_decay=cfg.training.weight_decay,
    grad_clip=cfg.training.grad_clip,
    grad_accum_steps=cfg.training.grad_accum_steps,
    early_stopping_patience=cfg.training.early_stopping_patience,
    use_amp=cfg.training.use_amp,
    scheduler=cfg.training.scheduler,
    class_weights=class_weights,
    num_classes=num_classes,
)

history = trainer.train()

In [ ]:
# Evaluate best checkpoint on test set
import torch.nn as nn
from utils import evaluate_model

best_ckpt = cfg.checkpoints_root / EXP_NAME / 'best.pt'
model = load_model(BaselineCNN(in_channels=3, num_classes=num_classes), str(best_ckpt), cfg.device)
criterion = nn.CrossEntropyLoss()

test_loss, test_acc, test_f1, test_bal, true_labels, pred_labels = evaluate_model(
    model, test_loader, cfg.device, criterion
)

print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test Macro-F1: {test_f1:.4f}  ← primary metric')
print(f'Balanced Acc: {test_bal:.4f}')

show_results(true_labels, pred_labels, cfg.classes, save_dir=str(cfg.results_root / EXP_NAME))